# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule
A page is worth reviewing if it already earns real traffic (average >=1 click/day over the prior
90 days) AND its search position is weak (worse than the eligible-population median). Both parts
are evidence-backed, not assumed: ML-06's signal audit confirmed a 13-point decline-rate gap
between weak- and strong-position pages within this eligible group (73.1% vs 60.3%, base rate
66.7%). GA4 coverage and click consistency were tested and dropped — GA4 coverage turned out to
be a client-level tracking artifact, not a page-level signal, and consistency's effect was too
thin to trust.

## Reason codes
- `visible_weak_position`   — clears the traffic floor, position worse than median
- `visible_strong_position` — clears the traffic floor, position at/better than median (not flagged)
- `below_traffic_floor`     — doesn't clear the floor, excluded from scoring entirely

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
from getpass import getpass
import os, duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
tbl = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {tbl}),
    decision AS (SELECT max_d - INTERVAL 30 DAY AS decision_date FROM bounds),
    prior AS (
        SELECT f.content_hash_id,
            AVG(f.gsc_clicks) AS avg_daily_clicks_prior,
            AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0) AS avg_position_prior,
            COUNT(*) AS n_days_prior
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date - INTERVAL 90 DAY AND f.report_date < d.decision_date
        GROUP BY f.content_hash_id
        HAVING COUNT(*) >= 30
    ),
    future AS (
        SELECT f.content_hash_id, AVG(f.gsc_clicks) AS avg_daily_clicks_future
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date AND f.report_date < d.decision_date + INTERVAL 30 DAY
        GROUP BY f.content_hash_id
    )
    SELECT p.*, fu.avg_daily_clicks_future
    FROM prior p JOIN future fu USING (content_hash_id)
""").df()

raw['avg_position_prior'] = raw['avg_position_prior'].fillna(100)
raw['is_declining_future'] = (
    (raw['avg_daily_clicks_future'] < 0.75 * raw['avg_daily_clicks_prior']) &
    (raw['avg_daily_clicks_prior'] >= 1.0)
).astype(int)

eligible = raw['avg_daily_clicks_prior'] >= 1.0
median_pos = raw.loc[eligible, 'avg_position_prior'].median()

scored = raw.loc[eligible].copy()
scored['score'] = scored['avg_position_prior']
scored['reason_code'] = np.where(scored['score'] > median_pos, 'visible_weak_position', 'visible_strong_position')
scored['actual_declined'] = scored['is_declining_future']
scored = scored.sort_values('score', ascending=False).reset_index(drop=True)
scored['rank'] = scored.index + 1

base_rate = scored['actual_declined'].mean()
p50 = scored['actual_declined'].iloc[:50].mean()
print(f'eligible pages: {len(scored):,}')
print(f'base rate: {base_rate:.3f}')
print(f'Precision@50: {p50:.3f}')

os.makedirs('work/outputs', exist_ok=True)
scored[['content_hash_id','score','reason_code','rank','actual_declined']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False
)

Paste your HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

eligible pages: 6,920
base rate: 0.667
Precision@50: 0.860


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [2]:
top20 = scored.head(20).copy()
top20['confidence'] = np.select(
    [top20['score'] >= top20['score'].quantile(0.9),
     top20['score'] >= median_pos * 1.5],
    ['high — far worse than eligible-population median position',
     'medium — moderately worse than median'],
    default='low — only marginally worse than median'
)
top20['would_be_wrong_if'] = np.where(
    top20['actual_declined'] == 1,
    'n/a — this page did decline, matches the rule',
    'flagged as at-risk but did NOT decline — check for a recent recovery not captured in the 90d prior window'
)

review = top20[['rank','content_hash_id','avg_position_prior','avg_daily_clicks_prior',
                 'reason_code','confidence','actual_declined','would_be_wrong_if']]
pd.set_option('display.max_colwidth', None)
review

,rank,content_hash_id,avg_position_prior,avg_daily_clicks_prior,reason_code,confidence,actual_declined,would_be_wrong_if
0,1,content_7b50820e76b1a006,52.629522,1.355556,visible_weak_position,high — far worse than eligible-population median position,1,"n/a — this page did decline, matches the rule"
1,2,content_b49f74ac9e13e0f5,47.453844,1.100000,visible_weak_position,high — far worse than eligible-population median position,0,flagged as at-risk but did NOT decline — check for a recent recovery not captured in the 90d prior window
2,3,content_580e7d0863f1bc6d,44.343632,1.100000,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
3,4,content_3e7917c6e649ea9f,43.198627,1.066667,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
4,5,content_ca27926935d855f0,42.750098,1.055556,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
5,6,content_da36aaa1d72bdad4,42.215869,1.277778,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
6,7,content_ea7474d92d9701c3,40.195290,2.611111,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
7,8,content_fbc4a78d5b8b8f0c,39.406748,1.411111,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
8,9,content_ba4867e06f07f3aa,39.281687,1.466667,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"
9,10,content_a11bd5663919f057,39.250706,1.733333,visible_weak_position,medium — moderately worse than median,1,"n/a — this page did decline, matches the rule"


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

2 of the top 20 flagged pages (10%) did not actually decline: `content_b49f74ac9e13e0f5` (rank 2,
position ~47.5, ~1.1 clicks/day) and `content_e7517465fc19e1cd` (rank 15, position ~38.0, ~1.4
clicks/day). Both sit well within the flagged range, not at the decision boundary near the
median — so the errors aren't a "borderline cases are noisy" pattern, they're pages with
genuinely weak position that simply didn't decline anyway. That's consistent with position being
a real but partial signal (ML-06's audit): it correlates with decline, it doesn't fully determine
it. A content team using this queue should expect roughly 1 in 10 top-priority flags to be a false
alarm, and should treat the reason code as "worth a look," not "confirmed problem."

Leakage check: scoring uses avg_position_prior only, compared against a threshold (median_pos)
computed from the same prior-window field. No future-window column enters the score or the
reason-code logic — confirmed by direct inspection of score_inputs above.

In [3]:
# Weak picks: top-20 rows the rule got wrong
weak_picks = top20[top20['actual_declined'] == 0]
print(f'weak picks in top 20: {len(weak_picks)} of 20')
print(weak_picks[['rank','content_hash_id','avg_position_prior','avg_daily_clicks_prior']])

# Leakage check: score/reason_code built ONLY from prior-window fields — confirm no future column touched
score_inputs = {'score': 'avg_position_prior', 'reason_code': 'avg_position_prior vs median_pos'}
future_cols_used_in_scoring = [c for c in score_inputs.values() if 'future' in c]
print('future-window fields used in scoring:', future_cols_used_in_scoring or 'NONE — confirmed clean')

weak picks in top 20: 2 of 20
    rank           content_hash_id  avg_position_prior  avg_daily_clicks_prior
1      2  content_b49f74ac9e13e0f5           47.453844                1.100000
14    15  content_e7517465fc19e1cd           37.993090                1.411111
future-window fields used in scoring: NONE — confirmed clean


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.